# Stellar Class — Optuna Hyperparameter Tuning
**Kaggle Playground Series S6E6**

Что делаем:
- Optuna автоматически перебирает параметры LightGBM
- Каждая попытка называется **trial**
- Optuna умная — не перебирает случайно, а учится на предыдущих попытках
- В конце берём лучшие параметры и обучаем финальную модель

## 1. Импорты

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)  # убираем спам от Optuna

RANDOM_STATE  = 42
N_FOLDS       = 5   # для тюнинга берём 5 — быстрее, Optuna компенсирует
N_TRIALS      = 50  # сколько комбинаций параметров попробовать
BASELINE_CV   = 0.93755  # наш лучший результат из v2

print('Libraries loaded ✅')
print(f'Optuna version: {optuna.__version__}')

Libraries loaded ✅
Optuna version: 4.9.0


## 2. Загрузка данных

In [2]:
train = pd.read_csv('../data/raw/train.csv')
test  = pd.read_csv('../data/raw/test.csv')

print(f'Train: {train.shape}')
print(f'Test:  {test.shape}')

Train: (577347, 12)
Test:  (247435, 11)


## 3. Feature Engineering (лучший набор из v2)

In [3]:
def engineer_features(df):
    df = df.copy()

    # Цветовые индексы v1
    df['u_g'] = df['u'] - df['g']
    df['g_r'] = df['g'] - df['r']
    df['r_i'] = df['r'] - df['i']
    df['i_z'] = df['i'] - df['z']
    df['u_r'] = df['u'] - df['r']
    df['u_z'] = df['u'] - df['z']
    df['g_z'] = df['g'] - df['z']
    df['g_i'] = df['g'] - df['i']

    # Новые комбинации v2
    df['r_z']  = df['r'] - df['z']
    df['u_i']  = df['u'] - df['i']
    df['r_g']  = df['r'] - df['g']
    df['i_g']  = df['i'] - df['g']
    df['z_r']  = df['z'] - df['r']

    # Трансформации redshift
    df['log1p_redshift'] = np.log1p(df['redshift'].clip(lower=0))
    df['redshift_sq']    = df['redshift'] ** 2

    # g_z фичи (была топ-1)
    df['g_z_sq']         = df['g_z'] ** 2
    df['g_z_x_redshift'] = df['g_z'] * df['log1p_redshift']
    df['g_z_x_u_g']      = df['g_z'] * df['u_g']

    # Interaction terms
    df['u_g_x_redshift'] = df['u_g'] * df['log1p_redshift']
    df['g_r_x_redshift'] = df['g_r'] * df['log1p_redshift']
    df['r_z_x_redshift'] = df['r_z'] * df['log1p_redshift']
    df['r_i_x_redshift'] = df['r_i'] * df['log1p_redshift']

    return df

train = engineer_features(train)
test  = engineer_features(test)

DROP_COLS    = ['id', 'class', 'alpha', 'delta', 'spectral_type', 'galaxy_population']
FEATURE_COLS = [c for c in train.columns if c not in DROP_COLS]

X      = train[FEATURE_COLS]
y      = train['class']
X_test = test[FEATURE_COLS]

le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

print(f'Фичей: {len(FEATURE_COLS)} ✅')
print(f'Классы: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}')

Фичей: 28 ✅
Классы: {'GALAXY': np.int64(0), 'QSO': np.int64(1), 'STAR': np.int64(2)}


## 4. Optuna — поиск лучших параметров

**Как работает Optuna:**
- Каждый trial = одна попытка с определёнными параметрами
- Optuna использует алгоритм TPE — умный байесовский поиск
- После каждой попытки обновляет своё "мнение" о том где искать дальше
- Намного эффективнее случайного перебора

**Пространство поиска** — диапазоны в которых Optuna ищет:

In [ ]:
def objective(trial):
    """Функция которую Optuna пытается максимизировать."""

    # Пространство поиска — Optuna сама выбирает значения в этих диапазонах
    params = {
        'objective':         'multiclass',
        'num_class':         3,
        'metric':            'multi_logloss',
        'verbose':           -1,
        'random_state':      RANDOM_STATE,

        # Параметры которые Optuna подбирает:
        'n_estimators':      trial.suggest_int('n_estimators', 500, 2000),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 31, 255),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }

    # Быстрая CV на 5 фолдах для оценки параметров
    skf    = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores = []

    for train_idx, val_idx in skf.split(X, y_encoded):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(period=-1)
            ]
        )

        preds  = np.argmax(model.predict_proba(X_val), axis=1)
        scores.append(balanced_accuracy_score(y_val, preds))

    return np.mean(scores)


print(f'Запускаем Optuna: {N_TRIALS} trials × {N_FOLDS} folds')
print(f'Примерное время: {N_TRIALS * N_FOLDS * 0.5:.0f}–{N_TRIALS * N_FOLDS * 1:.0f} минут')
print('Следи за прогрессом ниже...\n')

study = optuna.create_study(direction='maximize')  # максимизируем Balanced Accuracy
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✅ Optuna завершила поиск!')
print(f'Лучший CV Score: {study.best_value:.5f}')
print(f'Прирост vs baseline: {study.best_value - BASELINE_CV:+.5f}')

Запускаем Optuna: 50 trials × 5 folds
Примерное время: 125–250 минут
Следи за прогрессом ниже...



  0%|          | 0/50 [00:00<?, ?it/s]

## 5. Лучшие параметры

In [ ]:
print('Лучшие параметры найденные Optuna:')
print('─' * 40)
for param, value in study.best_params.items():
    print(f'  {param:25s}: {value}')

## 6. График прогресса Optuna

In [ ]:
# История trials
trials_df = study.trials_dataframe()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# График улучшения по trials
axes[0].plot(trials_df['number'], trials_df['value'], 'o', alpha=0.4, color='#4C72B0', markersize=4)
axes[0].plot(trials_df['number'], trials_df['value'].cummax(), color='#DD8452', linewidth=2, label='Лучший')
axes[0].axhline(BASELINE_CV, color='green', linestyle='--', label=f'Baseline {BASELINE_CV:.5f}')
axes[0].set_title('Прогресс Optuna по trials')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Balanced Accuracy')
axes[0].legend()

# Важность параметров
importances = optuna.importance.get_param_importances(study)
axes[1].barh(list(importances.keys()), list(importances.values()), color='#4C72B0')
axes[1].set_title('Важность параметров для Optuna')
axes[1].set_xlabel('Relative Importance')

plt.tight_layout()
plt.show()

## 7. Финальное обучение с лучшими параметрами

In [ ]:
# Берём лучшие параметры из Optuna
best_params = {
    'objective':   'multiclass',
    'num_class':   3,
    'metric':      'multi_logloss',
    'verbose':     -1,
    'random_state': RANDOM_STATE,
    **study.best_params  # подставляем найденные параметры
}

# Финальная CV с 8 фолдами для точной оценки
skf        = StratifiedKFold(n_splits=8, shuffle=True, random_state=RANDOM_STATE)
oof_preds  = np.zeros((len(train), 3))
test_preds = np.zeros((len(test), 3))
cv_scores  = []

print('Финальное обучение с лучшими параметрами (8-fold)...\n')

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_encoded)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]

    model = lgb.LGBMClassifier(**best_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(period=200)
        ]
    )

    val_proba = model.predict_proba(X_val)
    val_preds = np.argmax(val_proba, axis=1)
    score     = balanced_accuracy_score(y_val, val_preds)
    cv_scores.append(score)

    oof_preds[val_idx]  = val_proba
    test_preds         += model.predict_proba(X_test) / 8

    print(f'Fold {fold+1} | Balanced Accuracy: {score:.5f} | Деревьев: {model.best_iteration_}')

cv_mean = np.mean(cv_scores)
cv_std  = np.std(cv_scores)
delta   = cv_mean - BASELINE_CV

print(f'\n=== CV Score Optuna: {cv_mean:.5f} ± {cv_std:.5f} ===')
print(f'=== Baseline:        {BASELINE_CV:.5f} ===')
print(f'=== Дельта:          {delta:+.5f} === {"✅ Улучшение!" if delta > 0 else "❌ Хуже"}')

## 8. Сабмит

In [ ]:
test_classes = np.argmax(test_preds, axis=1)
test_labels  = le_target.inverse_transform(test_classes)

submission = pd.DataFrame({
    'id':    test['id'],
    'class': test_labels
})

submission.to_csv('../submissions/submission_v3_optuna.csv', index=False)

print('Сабмит сохранён ✅')
print(f'Файл: submissions/submission_v3_optuna.csv')
print('\nРаспределение предсказаний:')
print(submission['class'].value_counts())

## 9. Итоговая таблица прогресса

In [ ]:
print('=' * 55)
print(f'  Версия       │  CV Score  │  Что изменили')
print('─' * 55)
print(f'  v1 baseline  │  0.93721   │  LightGBM из коробки')
print(f'  v2 features  │  0.93755   │  Новые фичи + параметры')
print(f'  v3 optuna    │  {cv_mean:.5f}   │  Автоподбор параметров')
print('=' * 55)